In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, Optional, Tuple, Union, List

import numpy as np
import pandas as pd
import yfinance as yf

# Optional: pretty display in notebooks
try:
    from IPython.display import display
except Exception:  # pragma: no cover
    display = None


def _as_datetime(x):
    if x is None:
        return None
    return pd.to_datetime(x)


def _fetch_prices(
    tickers,
    start: str,
    end: str | None,
    auto_adjust: bool = True,
) -> pd.DataFrame:
    """
    Robust price fetcher for yfinance across:
      - single vs multiple tickers
      - MultiIndex ticker/field ordering
      - auto_adjust True/False

    Returns a DataFrame with columns = tickers, values = adjusted close series.
    """
    tickers = list(tickers)

    df = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        auto_adjust=auto_adjust,
        progress=False,
        group_by="ticker",
        threads=True,
    )

    if df is None or df.empty:
        return pd.DataFrame()

    # Decide which field we want
    # auto_adjust=True => "Close" is already adjusted
    wanted = "Close" if auto_adjust else "Adj Close"

    # --------
    # MultiIndex case
    # --------
    if isinstance(df.columns, pd.MultiIndex):
        lvl0 = df.columns.get_level_values(0)
        lvl1 = df.columns.get_level_values(1)

        # Case A: (ticker, field)
        if wanted in set(lvl1):
            px = df.xs(wanted, level=1, axis=1)
        # Case B: (field, ticker)
        elif wanted in set(lvl0):
            px = df.xs(wanted, level=0, axis=1)
        else:
            # Fallback: try Close then Adj Close
            for alt in ["Close", "Adj Close"]:
                if alt in set(lvl1):
                    px = df.xs(alt, level=1, axis=1)
                    break
                if alt in set(lvl0):
                    px = df.xs(alt, level=0, axis=1)
                    break
            else:
                raise KeyError(f"Could not find Close/Adj Close in columns: {df.columns}")

        px = px.copy()
        px.columns = [c.upper() for c in px.columns]  # normalize
        return px.dropna(how="all")

    # --------
    # Single-ticker (flat columns) case
    # --------
    if wanted in df.columns:
        s = df[wanted].copy()
    elif (not auto_adjust) and ("Close" in df.columns):
        # fallback if Adj Close missing
        s = df["Close"].copy()
    else:
        raise KeyError(f"Could not find '{wanted}' (or fallback) in columns: {df.columns.tolist()}")

    # If single ticker, return as DataFrame with that ticker name
    name = tickers[0].upper() if len(tickers) == 1 else "PRICE"
    return pd.DataFrame({name: s}).dropna(how="all")


def _returns(prices: pd.Series) -> pd.Series:
    r = prices.pct_change().replace([np.inf, -np.inf], np.nan)
    return r.dropna()


def _max_drawdown(prices: pd.Series) -> float:
    peak = prices.cummax()
    dd = prices / peak - 1.0
    return float(dd.min())


def _drawdown_series(prices: pd.Series) -> pd.Series:
    peak = prices.cummax()
    dd = prices / peak - 1.0
    return dd


def _ulcer_index(prices: pd.Series) -> float:
    dd = _drawdown_series(prices)
    # ulcer index uses drawdown in percent; square, average, sqrt
    dd_pct = (dd * 100.0).clip(upper=0)
    ui = float(np.sqrt(np.mean(np.square(dd_pct.values))))
    return ui


def _cagr(prices: pd.Series) -> float:
    if prices.shape[0] < 2:
        return np.nan
    start_val = float(prices.iloc[0])
    end_val = float(prices.iloc[-1])
    if start_val <= 0 or end_val <= 0:
        return np.nan
    days = (prices.index[-1] - prices.index[0]).days
    if days <= 0:
        return np.nan
    years = days / 365.25
    return float((end_val / start_val) ** (1 / years) - 1)


def _ann_vol(returns: pd.Series, periods_per_year: int = 252) -> float:
    if len(returns) < 2:
        return np.nan
    return float(returns.std(ddof=1) * np.sqrt(periods_per_year))


def _martin_ratio(prices: pd.Series, rf: float = 0.0) -> float:
    # Martin = (annualized return - rf)) / Ulcer Index
    ui = _ulcer_index(prices)
    if ui == 0 or np.isnan(ui):
        return np.nan
    ann = _cagr(prices)
    return float((ann - rf) / (ui / 100.0))  # ui was percent-based


def _time_underwater_stats(prices: pd.Series) -> Tuple[float, float, float, float]:
    """
    Returns:
      pct_days_underwater, avg_underwater_days, p95_underwater_days, max_underwater_days
    Underwater = price < prior peak.
    """
    if prices.shape[0] < 2:
        return (np.nan, np.nan, np.nan, np.nan)

    peak = prices.cummax()
    underwater = prices < peak

    # Identify contiguous underwater runs
    runs = []
    in_run = False
    run_start = None

    for dt, uw in underwater.items():
        if uw and not in_run:
            in_run = True
            run_start = dt
        elif (not uw) and in_run:
            # run ends at previous date; count business days in slice
            in_run = False
            run_end = dt
            # count rows between run_start and the day before run_end
            run_len = prices.loc[run_start:run_end].shape[0] - 1
            runs.append(max(run_len, 1))
            run_start = None

    if in_run and run_start is not None:
        run_len = prices.loc[run_start:].shape[0]
        runs.append(max(run_len, 1))

    pct = float(underwater.mean())
    if len(runs) == 0:
        return (pct, 0.0, 0.0, 0.0)

    runs_arr = np.array(runs, dtype=float)
    return (
        pct,
        float(np.mean(runs_arr)),
        float(np.percentile(runs_arr, 95)),
        float(np.max(runs_arr)),
    )


def _recovery_metrics(prices: pd.Series) -> Tuple[float, float, float, float]:
    """
    Period-local recovery notion:
      - Find the maximum drawdown peak->trough in the window.
      - Define "recovery" as trough -> reclaim that peak.
      Outputs:
        ttr_days, recovery_slope_annualized, recovery_efficiency, did_recover(0/1)
    """
    if prices.shape[0] < 5:
        return (np.nan, np.nan, np.nan, 0.0)

    peak = prices.cummax()
    dd = prices / peak - 1.0

    # Locate trough of worst drawdown
    trough_dt = dd.idxmin()
    trough_price = float(prices.loc[trough_dt])

    # Peak before trough that defines this drawdown
    pre = prices.loc[:trough_dt]
    if pre.shape[0] < 2:
        return (np.nan, np.nan, np.nan, 0.0)

    peak_dt = pre.idxmax()
    peak_price = float(prices.loc[peak_dt])

    if peak_price <= 0 or trough_price <= 0:
        return (np.nan, np.nan, np.nan, 0.0)

    max_dd = float((trough_price / peak_price) - 1.0)  # negative

    # Find first date after trough that reaches prior peak
    post = prices.loc[trough_dt:]
    recovered = post[post >= peak_price]
    if recovered.empty:
        # No recovery to old peak within window
        # Still can compute "slope" to end of window as partial recovery
        end_price = float(prices.iloc[-1])
        days = max((prices.index[-1] - trough_dt).days, 1)
        daily_slope = float(np.log(end_price / trough_price) / days) if end_price > 0 else np.nan
        ann_slope = float(np.exp(daily_slope * 252) - 1) if not np.isnan(daily_slope) else np.nan
        # Efficiency: partial return / abs(dd)
        partial_return = float((end_price / trough_price) - 1.0)
        eff = float(partial_return / abs(max_dd)) if max_dd != 0 else np.nan
        return (np.nan, ann_slope, eff, 0.0)

    rec_dt = recovered.index[0]
    ttr_days = max((rec_dt - trough_dt).days, 1)

    # log slope from trough to recovered peak
    daily_slope = float(np.log(peak_price / trough_price) / ttr_days)
    ann_slope = float(np.exp(daily_slope * 252) - 1)
    # recovery return from trough to peak
    rec_return = float((peak_price / trough_price) - 1.0)
    eff = float(rec_return / abs(max_dd)) if max_dd != 0 else np.nan
    return (float(ttr_days), ann_slope, eff, 1.0)


def _capital_efficiency_index(cagr: float, max_dd: float) -> float:
    """
    Capital Efficiency: CAGR / |MaxDD|
    Higher = better returns per unit of drawdown risk
    """
    if np.isnan(cagr) or np.isnan(max_dd) or max_dd >= 0:
        return np.nan
    return float(cagr / abs(max_dd))


def _volatility_drag_ratio(prices: pd.Series, returns: pd.Series) -> float:
    """
    Volatility Drag: geometric mean / arithmetic mean
    Ratio of actual compounded return to simple average return.
    Lower drag (closer to 1.0) is better.
    """
    if len(returns) < 2:
        return np.nan
    
    # Arithmetic mean return
    arith_mean = float(returns.mean())
    
    # Geometric mean: (ending / starting) ^ (1/n) - 1
    if prices.iloc[0] <= 0 or prices.iloc[-1] <= 0:
        return np.nan
    
    total_return = float(prices.iloc[-1] / prices.iloc[0]) - 1.0
    n_periods = len(returns)
    geom_mean = float((1 + total_return) ** (1 / n_periods) - 1)
    
    if arith_mean == 0:
        return np.nan
    return float(geom_mean / arith_mean)


def _pct_time_near_highs(prices: pd.Series, threshold_pct: float = 0.05) -> float:
    """
    % Time Near Highs: Percentage of days within threshold_pct of rolling peak
    threshold_pct: e.g., 0.05 for within 5% of peak
    """
    if prices.shape[0] < 2:
        return np.nan
    
    peak = prices.cummax()
    pct_below = (prices / peak - 1.0)  # negative values = below peak
    near_high = pct_below >= -threshold_pct
    
    return float(near_high.mean())


def _trend_persistence(returns: pd.Series) -> float:
    """
    Trend Persistence: % of days where return has same sign as previous day
    Higher = stronger momentum/trend, Lower = more mean reversion
    """
    if len(returns) < 2:
        return np.nan
    
    sign_change = np.sign(returns.values[:-1]) == np.sign(returns.values[1:])
    # Only count where neither is zero
    valid = (returns.values[:-1] != 0) & (returns.values[1:] != 0)
    
    if valid.sum() == 0:
        return np.nan
    
    return float(sign_change[valid].mean())


def _expected_shortfall(returns: pd.Series, percentile: float = 5.0) -> float:
    """
    Expected Shortfall (CVaR): Average of worst X% of returns
    percentile: e.g., 5.0 for worst 5%
    """
    if len(returns) < 10:
        return np.nan
    
    threshold = np.percentile(returns.values, percentile)
    worst = returns[returns <= threshold]
    
    if len(worst) == 0:
        return np.nan
    
    return float(worst.mean())


def _capture_ratio(etf_prices: pd.Series, bench_prices: pd.Series, side: str) -> float:
    """
    Upside/Downside capture within window:
      - Uses daily returns
      - For upside: only days benchmark return > 0
      - For downside: only days benchmark return < 0
      Ratio of compounded returns over those selected days
    """
    etf_r = _returns(etf_prices)
    bmk_r = _returns(bench_prices)

    aligned = pd.concat([etf_r.rename("etf"), bmk_r.rename("bmk")], axis=1).dropna()
    
    if aligned.empty:
        return np.nan

    if side == "up":
        sel = aligned[aligned["bmk"] > 0]
    elif side == "down":
        sel = aligned[aligned["bmk"] < 0]
    else:
        raise ValueError("side must be 'up' or 'down'")

    if sel.empty:
        return np.nan

    etf_comp = float(np.prod(1.0 + sel["etf"].values) - 1.0)
    bmk_comp = float(np.prod(1.0 + sel["bmk"].values) - 1.0)
    if bmk_comp == 0:
        return np.nan
    return float(etf_comp / bmk_comp)


# --- Helper to normalize period values ---
def _normalize_period(val: Union[Tuple[str, Optional[str]], List[Optional[str]]]) -> Tuple[str, Optional[str]]:
    """
    Accepts period value as:
      - [start, end]
      - [start, None]
      - [start, end, trough]  (third element ignored)
    Returns (start, end) with end possibly None.
    """
    if isinstance(val, (list, tuple)):
        if len(val) >= 2:
            return val[0], val[1]
        raise ValueError("Period value must have at least [start, end_or_None]")
    raise ValueError(f"Unsupported period value type: {type(val)}")


def _period_metrics_for_series(
    etf_prices: pd.Series,
    bench_prices: Optional[pd.Series] = None,
    rf: float = 0.0,
) -> Dict[str, float]:
    r = _returns(etf_prices)

    maxdd = _max_drawdown(etf_prices)
    ui = _ulcer_index(etf_prices)
    cagr = _cagr(etf_prices)
    vol = _ann_vol(r)
    martin = _martin_ratio(etf_prices, rf=rf)

    pct_uw, avg_uw, p95_uw, max_uw = _time_underwater_stats(etf_prices)
    ttr, rec_slope, rec_eff, did_rec = _recovery_metrics(etf_prices)

    # New metrics
    cap_eff = _capital_efficiency_index(cagr, maxdd)
    vol_drag = _volatility_drag_ratio(etf_prices, r)
    pct_near_highs = _pct_time_near_highs(etf_prices)
    trend_pers = _trend_persistence(r)
    cvar = _expected_shortfall(r)

    # Period-level relative metrics (not rolling - calculated over full period)
    periods_per_year = 252
    period_corr = np.nan
    period_beta = np.nan
    period_alpha_per = np.nan
    period_r2 = np.nan
    te_ann = np.nan
    ir = np.nan
    cond_corr_up = np.nan
    cond_corr_down = np.nan
    period_alpha_ann = np.nan

    if bench_prices is not None:
        bmk_r = _returns(bench_prices)
        # Align returns
        aligned = pd.concat([r.rename("etf"), bmk_r.rename("bmk")], axis=1).dropna()
        if len(aligned) >= 10:  # Need minimum data points
            etf_r = aligned["etf"]
            bmk_r = aligned["bmk"]
            
            # Full period correlation
            period_corr = float(etf_r.corr(bmk_r))
            
            # Full period beta/alpha/R2 using simple linear regression
            # beta = cov(etf, bmk) / var(bmk)
            cov_xy = etf_r.cov(bmk_r)
            var_x = bmk_r.var()
            
            if var_x > 0:
                period_beta = float(cov_xy / var_x)
                # alpha = mean(etf) - beta * mean(bmk)
                period_alpha_per = float(etf_r.mean() - period_beta * bmk_r.mean())
                # R² = correlation²
                period_r2 = float(period_corr ** 2)
                
                # Annualized alpha (simple scaling - for daily returns)
                period_alpha_ann = float(period_alpha_per * periods_per_year)
                
                # Tracking error: std(etf - bmk) annualized
                active_returns = etf_r - bmk_r
                te_ann = float(active_returns.std(ddof=1) * np.sqrt(periods_per_year))
                
                # Information ratio: annualized_alpha / tracking_error
                if te_ann > 0:
                    ir = float(period_alpha_ann / te_ann)
            
            # Conditional correlations (full period)
            # Correlation when benchmark is up
            up_mask = bmk_r > 0
            if up_mask.sum() >= 10:
                cond_corr_up = float(etf_r[up_mask].corr(bmk_r[up_mask]))
            
            # Correlation when benchmark is down
            down_mask = bmk_r < 0
            if down_mask.sum() >= 10:
                cond_corr_down = float(etf_r[down_mask].corr(bmk_r[down_mask]))

    out = {
        "CAGR": cagr,
        "AnnVol": vol,
        "MaxDD": maxdd,
        "UlcerIndex": ui,
        "Martin": martin,
        "CapitalEfficiency": cap_eff,
        "VolatilityDrag": vol_drag,
        "PctTimeNearHighs": pct_near_highs,
        "TrendPersistence": trend_pers,
        "ExpectedShortfall": cvar,
        "PctDaysUnderwater": pct_uw,
        "AvgUnderwaterDays": avg_uw,
        "P95UnderwaterDays": p95_uw,
        "MaxUnderwaterDays": max_uw,
        "TTR_Days": ttr,
        "RecoverySlope_Ann": rec_slope,
        "RecoveryEfficiency": rec_eff,
        "RecoveredToPeak_inWindow": did_rec,
        # Period-level relative metrics (vs benchmark)
        "PeriodCorr": period_corr,
        "PeriodBeta": period_beta,
        "PeriodAlpha_per_period": period_alpha_per,
        "PeriodAlpha_Ann": period_alpha_ann,
        "PeriodR2": period_r2,
        "TrackingError_Ann": te_ann,
        "InformationRatio": ir,
        "ConditionalCorr_Up": cond_corr_up,
        "ConditionalCorr_Down": cond_corr_down,
    }

    if bench_prices is not None:
        out["UpsideCapture"] = _capture_ratio(etf_prices, bench_prices, "up")
        out["DownsideCapture"] = _capture_ratio(etf_prices, bench_prices, "down")

    return out


def _format_styler(df: pd.DataFrame) -> "pd.io.formats.style.Styler":
    pct_cols = [c for c in df.columns if c in {
        "CAGR", "AnnVol", "MaxDD", "PctTimeNearHighs", "TrendPersistence",
        "RecoverySlope_Ann", "PeriodAlpha_per_period", "PeriodAlpha_Ann", "TrackingError_Ann"
    }]
    ratio_cols = [c for c in df.columns if "Capture" in c or c in {
        "Martin", "RecoveryEfficiency", "CapitalEfficiency", "VolatilityDrag",
        "InformationRatio", "PeriodR2"
    }]
    corr_beta_cols = [c for c in df.columns if c in {
        "PeriodCorr", "ConditionalCorr_Up", "ConditionalCorr_Down", "PeriodBeta"
    }]
    num_cols = [c for c in df.columns if c not in pct_cols + ratio_cols + corr_beta_cols]

    sty = df.style

    # Formats
    fmt = {c: "{:.2%}" for c in pct_cols}
    fmt.update({c: "{:.3f}" for c in ratio_cols})
    fmt.update({c: "{:.3f}" for c in corr_beta_cols})
    fmt.update({c: "{:.2f}" for c in num_cols if c not in fmt})

    # Special-case some columns
    if "UlcerIndex" in df.columns:
        fmt["UlcerIndex"] = "{:.2f}"
    if "PctDaysUnderwater" in df.columns:
        fmt["PctDaysUnderwater"] = "{:.1%}"
    if "ExpectedShortfall" in df.columns:
        fmt["ExpectedShortfall"] = "{:.2%}"
    for c in ["AvgUnderwaterDays", "P95UnderwaterDays", "MaxUnderwaterDays", "TTR_Days"]:
        if c in df.columns:
            fmt[c] = "{:.0f}"
    if "RecoveredToPeak_inWindow" in df.columns:
        fmt["RecoveredToPeak_inWindow"] = "{:.0f}"
    
    # Add DataAvailability column formatting
    if "DataAvailability" in df.columns:
        fmt["DataAvailability"] = "{:.1%}"

    sty = sty.format(fmt, na_rep="—")

    # Only apply gradient to columns that have at least one non-NaN value
    gradient_cols = []
    for c in df.columns:
        if c != "DataAvailability" and df[c].notna().any():
            gradient_cols.append(c)
    
    if gradient_cols:
        sty = sty.background_gradient(subset=gradient_cols)

    if "MaxDD" in df.columns:
        sty = sty.map(lambda v: "font-weight:600" if pd.notna(v) and v < 0 else "", subset=["MaxDD"])

    return sty


def analyze_regimes(
    tickers: Union[str, Iterable[str]],
    periods: Dict[str, Union[Tuple[str, Optional[str]], List[Optional[str]]]],
    benchmark: Optional[str] = "VTI",
    rf: float = 0.0,
    auto_adjust: bool = True,
    display_results: bool = True,
) -> Dict[str, pd.DataFrame]:
    """
    Analyze ETF behavior over multiple named time windows.

    periods: dict mapping name -> [start, end] or [start, end, trough].
    The third element (trough) is ignored; only start/end are used.

    **KEY FEATURE**: Handles ETFs with different inception dates by:
    - Using whatever data is available within each period
    - Adding a "DataAvailability" column showing % of period covered
    - Computing metrics based on the actual overlapping dates

    Usage:
      analyze_regimes(["QQQ","SPY","USMV"], periods=periods_dict, benchmark="VTI")

    Args:
      tickers: one ticker or list of tickers to analyze
      periods: dict mapping name -> (start, end). end can be None to mean "today"
      benchmark: benchmark ticker for upside/downside capture (None disables)
      rf: risk-free rate used in Martin ratio (annual, e.g. 0.02)
      auto_adjust: pass through to yfinance download
      display_results: if True and in IPython, displays styled tables

    Returns:
      dict mapping period name -> DataFrame (rows=tickers, cols=metrics + DataAvailability)
    """
    if isinstance(tickers, str):
        tick_list = [tickers]
    else:
        tick_list = list(tickers)

    results: Dict[str, pd.DataFrame] = {}

    for pname, pval in periods.items():
        # Normalize period value to (start, end)
        pstart, pend = _normalize_period(pval)

        pstart_dt = _as_datetime(pstart)
        pend_dt = _as_datetime(pend)

        if pstart_dt is None:
            raise ValueError(f"Period '{pname}' missing start date.")
        if pend_dt is not None and pend_dt <= pstart_dt:
            raise ValueError(f"Period '{pname}' has end <= start.")

        # Fetch prices for ETFs + benchmark in one call when possible
        fetch_list = tick_list.copy()
        bmk = None
        if benchmark:
            fetch_list = sorted(set(fetch_list + [benchmark]))
            bmk = benchmark

        # Fetch with wider date range to capture all possible data
        px = _fetch_prices(
            fetch_list, 
            start=str(pstart_dt.date()), 
            end=(None if pend_dt is None else str(pend_dt.date())), 
            auto_adjust=auto_adjust
        )

        if px.empty:
            continue

        # Calculate full period trading days for availability calculation
        full_period_end = pend_dt if pend_dt is not None else px.index.max()
        full_period_mask = (px.index >= pstart_dt) & (px.index <= full_period_end)
        full_period_days = full_period_mask.sum()

        rows = []
        idx = []

        bench_series = px[bmk].dropna() if (bmk and bmk in px.columns) else None

        for t in tick_list:
            if t not in px.columns:
                continue
            
            # Get ticker series and filter to period
            s_full = px[t].dropna()
            
            # Find actual overlap with requested period
            s = s_full.loc[pstart_dt: full_period_end]
            
            if len(s) < 10:  # Need minimum data for meaningful analysis
                continue
            
            # Calculate data availability
            actual_days = len(s)
            data_availability = actual_days / full_period_days if full_period_days > 0 else 0.0
            
            # Get benchmark data aligned to ETF dates
            b = None
            if bench_series is not None:
                b = bench_series.reindex(s.index).dropna()
                # Re-align s to b to ensure matching dates
                if len(b) > 0:
                    common_idx = s.index.intersection(b.index)
                    s = s.loc[common_idx]
                    b = b.loc[common_idx]

            if len(s) < 10:
                continue

            metrics = _period_metrics_for_series(
                s, 
                bench_prices=b if b is not None else None, 
                rf=rf
            )
            
            # Add data availability to metrics
            metrics["DataAvailability"] = data_availability
            
            rows.append(metrics)
            idx.append(t)

        if not rows:
            continue

        df = pd.DataFrame(rows, index=idx)

        # Sort columns with DataAvailability first
        col_order = [
            "DataAvailability",  # Show this first so users know coverage
            "CAGR", "AnnVol", "Martin", "CapitalEfficiency",
            "MaxDD", "UlcerIndex",
            "VolatilityDrag", "PctTimeNearHighs", "TrendPersistence",
            "PctDaysUnderwater", "AvgUnderwaterDays", "P95UnderwaterDays", "MaxUnderwaterDays",
            "TTR_Days", "RecoverySlope_Ann", "RecoveryEfficiency", "RecoveredToPeak_inWindow",
            "ExpectedShortfall",
            # Rolling analytics
            "RollingCorr", "ConditionalCorr_Up", "ConditionalCorr_Down",
            "RollingBeta", "RollingR2",
            "RollingAlpha_per_period", "RollingAlpha_Ann",
            "TrackingError_Ann", "InformationRatio",
            # Capture
            "UpsideCapture", "DownsideCapture",
        ]
        df = df[[c for c in col_order if c in df.columns] + [c for c in df.columns if c not in col_order]]

        results[pname] = df

        if display_results and display is not None:
            title = pd.DataFrame({
                "Period": [pname], 
                "Start": [str(pstart_dt.date())], 
                "End": [("today" if pend_dt is None else str(full_period_end.date()))]
            })
            display(title)
            display(_format_styler(df))

    return results


# -------------------------
# Rolling analytics helpers
# -------------------------
def _to_returns(px: pd.Series, freq: str = "D") -> pd.Series:
    """
    Convert price series to returns.
    freq:
      - "D": daily returns
      - "W": weekly returns (Fri close)
      - "M": monthly returns
    """
    px = px.dropna()
    if freq == "D":
        r = px.pct_change()
    elif freq == "W":
        r = px.resample("W-FRI").last().pct_change()
    elif freq == "M":
        r = px.resample("M").last().pct_change()
    else:
        raise ValueError("freq must be one of {'D','W','M'}")
    return r.replace([np.inf, -np.inf], np.nan).dropna()


def rolling_corr(etf_r: pd.Series, bmk_r: pd.Series, window: int) -> pd.Series:
    df = pd.concat([etf_r.rename("etf"), bmk_r.rename("bmk")], axis=1).dropna()
    return df["etf"].rolling(window).corr(df["bmk"])


def rolling_beta_alpha(etf_r: pd.Series, bmk_r: pd.Series, window: int) -> pd.DataFrame:
    """
    OLS in each rolling window:
      etf = alpha + beta*bmk + eps

    Returns DataFrame columns: ['alpha', 'beta', 'r2']
    alpha is per-period (daily/weekly/monthly) alpha, NOT annualized.
    """
    df = pd.concat([etf_r.rename("etf"), bmk_r.rename("bmk")], axis=1).dropna()

    # Rolling means
    mx = df["bmk"].rolling(window).mean()
    my = df["etf"].rolling(window).mean()

    # Rolling cov/var
    cov_xy = df["bmk"].rolling(window).cov(df["etf"])
    var_x  = df["bmk"].rolling(window).var()

    beta = cov_xy / var_x
    alpha = my - beta * mx

    # Rolling R^2: correlation squared is simpler and more stable
    corr = df["etf"].rolling(window).corr(df["bmk"])
    r2 = corr ** 2

    out = pd.DataFrame({"alpha": alpha, "beta": beta, "r2": r2})
    return out


def annualize_alpha(alpha_per_period: pd.Series, periods_per_year: int) -> pd.Series:
    """
    Convert per-period alpha (e.g., daily) into annualized additive approximation.
    For small alpha this is fine. If you prefer compounding, use exp/log.
    """
    return alpha_per_period * periods_per_year


def tracking_error(etf_r: pd.Series, bmk_r: pd.Series, window: int, periods_per_year: int) -> pd.Series:
    """
    Rolling annualized tracking error: std(etf - bmk) * sqrt(periods_per_year)
    """
    df = pd.concat([etf_r.rename("etf"), bmk_r.rename("bmk")], axis=1).dropna()
    active = (df["etf"] - df["bmk"])
    return active.rolling(window).std(ddof=1) * np.sqrt(periods_per_year)


def information_ratio(
    alpha_per_period: pd.Series,
    te_annualized: pd.Series,
    periods_per_year: int
) -> pd.Series:
    """
    Rolling Information Ratio:
      IR = annualized_alpha / annualized_tracking_error
    alpha_per_period should align with te_annualized index.
    """
    ann_alpha = annualize_alpha(alpha_per_period, periods_per_year)
    return ann_alpha / te_annualized.replace(0, np.nan)


def conditional_corr(
    etf_r: pd.Series,
    bmk_r: pd.Series,
    window: int,
    condition: str = "down"
) -> pd.Series:
    """
    Rolling correlation computed only on observations inside each window
    where benchmark return is <0 (down) or >0 (up).

    Note: This is more expensive than plain rolling corr because the set
    of points changes per window. For typical ETF daily data it's fine.
    """
    df = pd.concat([etf_r.rename("etf"), bmk_r.rename("bmk")], axis=1).dropna()

    def _corr_in_window(idx: pd.Index) -> float:
        x = df.loc[idx]
        if condition == "down":
            sub = x[x["bmk"] < 0]
        elif condition == "up":
            sub = x[x["bmk"] > 0]
        else:
            raise ValueError("condition must be 'down' or 'up'")
        if len(sub) < max(10, window // 5):  # require some minimum points
            return np.nan
        return float(sub["etf"].corr(sub["bmk"]))

    # Apply rolling to one column, use its index to get full DataFrame slice
    return df["etf"].rolling(window).apply(lambda w: _corr_in_window(w.index), raw=False)


In [2]:
periods = {
    "Full History": [
      "2018-05-01", None
    ],
  }



In [3]:
additional_periods = {
    "DotCom Bubble Drawdown": [
      "2001-06-02",
      "2002-10-09"
    ],
    "Lost Decade Recovery": [
      "2002-10-09",
      "2007-06-13"
    ],
    "GFC Drawdown": [
      "2007-06-13",
      "2009-03-09"
    ],
    "2009–2015 Bull": [
      "2009-03-09",
      "2015-05-23"
    ],
    "2015-2016 Selloff Drawdown": [
      "2015-05-23",
      "2016-02-11",
    ],
    "2016-2017 Bull": [
      "2016-02-11",
      "2017-12-26"
    ],
    "Volmageddon Drawdown": [
      "2017-12-26",
      "2018-12-24",
    ],
    "2019-2020 Bull": [
      "2018-12-24",
      "2020-01-19"
    ],
    "Covid Drawdown": [
      "2020-01-19",
      "2020-03-23"
    ],
    "2021 Bull": [
      "2020-03-23",
      "2021-12-03"
    ],
    "2022 Rate Hike Drawdown": [
      "2021-12-03",
      "2022-10-12",
    ],
    "Modern Bull": [
      "2022-10-12",
      None
    ]
}

In [14]:
ticker_list = ['VOO', 'SPMO', 'VFMO', 'SPHQ', 'QUAL', 'QLD', 'DBMF', 'SSO', 'KMLM', 'UPRO']
# ['VFMO', 'SPMO', 'JMOM', 'SGRT', 'FMTM', 'SAMM', 'ATFV']

In [15]:
analyze_regimes(ticker_list, periods=periods, benchmark="VOO", rf=0.0)
print("done")

,Period,Start,End
0,Full History,2018-05-01,today


,DataAvailability,CAGR,AnnVol,Martin,CapitalEfficiency,MaxDD,UlcerIndex,VolatilityDrag,PctTimeNearHighs,TrendPersistence,PctDaysUnderwater,AvgUnderwaterDays,P95UnderwaterDays,MaxUnderwaterDays,TTR_Days,RecoverySlope_Ann,RecoveryEfficiency,RecoveredToPeak_inWindow,ExpectedShortfall,ConditionalCorr_Up,ConditionalCorr_Down,TrackingError_Ann,InformationRatio,UpsideCapture,DownsideCapture,PeriodCorr,PeriodBeta,PeriodAlpha_per_period,PeriodAlpha_Ann,PeriodR2
VOO,100.0%,14.96%,19.39%,1.894,0.440,-33.99%,7.90,0.881,65.60%,50.65%,84.4%,12,41,488,140,111.22%,1.515,1,-2.91%,1.000,1.000,0.00%,—,1.000,1.000,1.000,1.000,0.00%,0.00%,1.000
SPMO,100.0%,18.75%,21.69%,2.344,0.606,-30.95%,8.00,0.880,59.67%,50.00%,87.3%,14,59,487,107,139.20%,1.448,1,-3.19%,0.842,0.866,9.11%,0.383,1.269,1.000,0.908,1.016,0.01%,3.49%,0.824
VFMO,100.0%,15.35%,23.65%,1.418,0.417,-36.77%,10.82,0.836,46.21%,49.70%,90.9%,23,113,560,129,144.88%,1.582,1,-3.50%,0.773,0.826,11.41%,0.013,2.894,1.000,0.878,1.071,0.00%,0.15%,0.771
SPHQ,100.0%,15.02%,19.24%,2.029,0.475,-31.59%,7.40,0.883,68.34%,49.00%,86.1%,12,38,392,134,104.23%,1.462,1,-2.85%,0.949,0.950,4.76%,0.130,0.717,1.000,0.970,0.962,0.00%,0.62%,0.940
QUAL,100.0%,13.89%,19.55%,1.598,0.408,-34.06%,8.69,0.872,64.61%,50.10%,86.7%,13,40,489,142,109.37%,1.516,1,-2.92%,0.969,0.976,3.47%,-0.227,0.994,1.000,0.984,0.992,-0.00%,-0.79%,0.969
QLD,100.0%,29.84%,47.76%,1.233,0.469,-63.68%,24.20,0.695,34.10%,49.70%,90.5%,21,90,629,513,64.47%,2.754,1,-7.08%,0.890,0.887,30.41%,0.036,190651.321,1.001,0.936,2.305,0.00%,1.10%,0.875
DBMF,87.2%,9.25%,12.45%,1.078,0.454,-20.39%,8.58,0.919,49.71%,48.61%,93.8%,32,96,793,1001,5.91%,1.256,1,-1.92%,0.008,0.142,21.49%,0.360,0.002,0.654,0.185,0.115,0.03%,7.74%,0.034
SSO,100.0%,21.79%,38.66%,1.232,0.367,-59.34%,17.69,0.724,45.86%,50.75%,87.9%,15,53,541,163,302.02%,2.459,1,-5.83%,0.999,0.999,19.29%,-0.224,3619.127,1.001,1.000,1.993,-0.02%,-4.31%,0.999
KMLM,67.4%,6.54%,14.77%,0.365,0.211,-31.01%,17.92,0.853,23.58%,51.34%,95.5%,45,139,882,—,16.09%,0.643,0,-2.33%,-0.225,-0.078,23.70%,0.391,-0.000,-0.464,-0.135,-0.120,0.04%,9.27%,0.018
UPRO,100.0%,25.08%,57.87%,0.901,0.326,-76.82%,27.84,0.568,33.60%,50.50%,90.6%,20,66,615,291,254.61%,4.313,1,-8.74%,0.999,0.999,38.49%,-0.204,11501000.930,1.001,1.000,2.984,-0.03%,-7.84%,0.999


done
